# 🫘 Chronic Kidney Disease (CKD) Prediction - Model Training
## Explainable AI (XAI) Based Multi-Disease Risk Prediction System
### Dataset: UCI CKD Dataset (400 samples, 24 features)

**Target:** `classification` → ckd / notckd (binarized)  
**Models:** Logistic Regression, Decision Tree, Random Forest, XGBoost  
**XAI:** SHAP  
**Output:** `kidney_model.pkl`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os, pickle
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, confusion_matrix, roc_auc_score, roc_curve,
    f1_score, precision_score, recall_score, classification_report
)
import shap

os.makedirs('../backend/models', exist_ok=True)
print('✅ Libraries loaded')

## 1️⃣ Load & Explore Data

In [ ]:
df = pd.read_csv('kidney_disease.csv')  # Place in notebooks/ folder
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

In [ ]:
print('Missing values:')
print(df.isnull().sum().sort_values(ascending=False))
print('\nTarget value counts:')
print(df['classification'].value_counts())

## 2️⃣ Preprocessing

In [ ]:
# Drop id
df = df.drop(columns=['id'], errors='ignore')

# Fix classification target (handle 'ckd\t' encoding issue)
df['classification'] = df['classification'].str.strip()
df['target'] = (df['classification'] == 'ckd').astype(int)
df = df.drop(columns=['classification'])

print('Target counts after fix:')
print(df['target'].value_counts())

# Encode object columns (categorical features)
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f'\nCategorical columns: {cat_cols}')

encoders = {}
for col in cat_cols:
    df[col] = df[col].astype(str).str.strip()
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

# Convert numeric-looking object cols
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = pd.to_numeric(df[col], errors='coerce')

FEATURE_NAMES = [c for c in df.columns if c != 'target']
X = df[FEATURE_NAMES]
y = df['target']

print(f'\nFeatures ({len(FEATURE_NAMES)}): {FEATURE_NAMES}')

In [ ]:
# Impute missing values
imputer = SimpleImputer(strategy='median')
X_imp = pd.DataFrame(imputer.fit_transform(X), columns=FEATURE_NAMES)

print('Missing after imputation:', X_imp.isnull().sum().sum())

# Correlation heatmap
plt.figure(figsize=(14, 12))
corr_df = X_imp.copy(); corr_df['target'] = y.values
mask = np.triu(np.ones_like(corr_df.corr(), dtype=bool))
sns.heatmap(corr_df.corr(), mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.4, square=True, annot_kws={'size':6})
plt.title('CKD Feature Correlation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../backend/models/kidney_corr.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_imp, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = pd.DataFrame(scaler.fit_transform(X_train), columns=FEATURE_NAMES)
X_test_sc  = pd.DataFrame(scaler.transform(X_test), columns=FEATURE_NAMES)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')

## 3️⃣ Model Training

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(C=1.0, max_iter=3000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=6, min_samples_split=5, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=300, max_depth=10,
                                                   min_samples_leaf=2, random_state=42, n_jobs=-1),
    'XGBoost':             XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=5,
                                          subsample=0.8, colsample_bytree=0.8,
                                          use_label_encoder=False, eval_metric='logloss', random_state=42),
}

results = []
trained_models = {}

for name, model in models.items():
    X_tr = X_train_sc if name == 'Logistic Regression' else X_train
    X_te = X_test_sc  if name == 'Logistic Regression' else X_test
    X_cv = X_train_sc if name == 'Logistic Regression' else X_train

    cv_scores = cross_val_score(model, X_cv, y_train, cv=cv, scoring='accuracy')
    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:,1]

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    f1  = f1_score(y_test, y_pred)

    trained_models[name] = {'model': model, 'X_test': X_te, 'X_train': X_tr}
    results.append({'Model': name, 'CV Accuracy': cv_scores.mean(),
                    'Test Accuracy': acc, 'AUC-ROC': auc, 'F1 Score': f1,
                    'Precision': precision_score(y_test, y_pred),
                    'Recall': recall_score(y_test, y_pred)})
    print(f'✅ {name}: Acc={acc:.4f} | AUC={auc:.4f} | F1={f1:.4f}')

results_df = pd.DataFrame(results).sort_values('AUC-ROC', ascending=False)
results_df

In [ ]:
# Plots
plt.figure(figsize=(8,6))
for name, data in trained_models.items():
    fpr, tpr, _ = roc_curve(y_test, data['model'].predict_proba(data['X_test'])[:,1])
    auc = roc_auc_score(y_test, data['model'].predict_proba(data['X_test'])[:,1])
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', linewidth=2)
plt.plot([0,1],[0,1],'k--')
plt.xlabel('FPR'); plt.ylabel('TPR')
plt.title('ROC Curves - Kidney Disease', fontsize=14, fontweight='bold')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../backend/models/kidney_roc.png', dpi=150, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, (name, data) in zip(axes, trained_models.items()):
    cm = confusion_matrix(y_test, data['model'].predict(data['X_test']))
    sns.heatmap(cm, annot=True, fmt='d', ax=ax, cmap='Greens',
                xticklabels=['Not CKD','CKD'], yticklabels=['Not CKD','CKD'])
    ax.set_title(name, fontweight='bold')
plt.suptitle('Confusion Matrices - Kidney Disease', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../backend/models/kidney_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

## 4️⃣ SHAP Explainability

In [ ]:
best_name = results_df.iloc[0]['Model']
best_model = trained_models[best_name]['model']
X_test_best = trained_models[best_name]['X_test']
X_train_best = trained_models[best_name]['X_train']

print(f'🏆 Best Model: {best_name}')

if best_name in ['XGBoost','Random Forest','Decision Tree']:
    explainer = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(X_test_best)
    shap_vals = shap_values[1] if isinstance(shap_values, list) else shap_values
else:
    explainer = shap.LinearExplainer(best_model, X_train_best)
    shap_vals = explainer.shap_values(X_test_best)

plt.figure(figsize=(10,6))
shap.summary_plot(shap_vals, X_test_best, feature_names=FEATURE_NAMES, show=False)
plt.title(f'SHAP Summary - {best_name} (Kidney Disease)', fontweight='bold')
plt.tight_layout()
plt.savefig('../backend/models/kidney_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

plt.figure(figsize=(8,5))
shap.summary_plot(shap_vals, X_test_best, feature_names=FEATURE_NAMES, plot_type='bar', show=False)
plt.title(f'SHAP Feature Importance - {best_name}', fontweight='bold')
plt.tight_layout()
plt.savefig('../backend/models/kidney_shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 5️⃣ Save Model

In [ ]:
artifact = {
    'model': best_model,
    'model_name': best_name,
    'scaler': scaler,
    'imputer': imputer,
    'feature_names': FEATURE_NAMES,
    'model_results': results_df.to_dict('records'),
    'disease': 'kidney',
    'target_col': 'target',
    'classes': ['Not CKD', 'Chronic Kidney Disease'],
    'threshold': 0.5,
    'encoders': encoders
}

with open('../backend/models/kidney_model.pkl', 'wb') as f:
    pickle.dump(artifact, f)

print(f'✅ Saved → backend/models/kidney_model.pkl')
print(f'🏆 Best: {best_name}')
print(results_df.to_string(index=False))